# LSTM Movie Reviews

In [128]:
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn

import string
import nltk
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm  # Import tqdm for progress bars

In [129]:
# Check that MPS is available on Apple M series chip
if not torch.backends.mps.is_available():
    if not torch.backends.mps.is_built():
        print("MPS not available because the current PyTorch install was not "
              "built with MPS enabled.")
    else:
        print("MPS not available because the current MacOS version is not 12.3+ "
              "and/or you do not have an MPS-enabled device on this machine.")

else:
    device = torch.device("mps")
    print("MPS is available on this machine.")

MPS is available on this machine.


In [130]:
df_all_data = pd.read_csv('datasets/IMDB_Dataset.csv')

df_all_data#.sample(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


# Split dataset

In [131]:
df_train, df_test = train_test_split(df_all_data, test_size=0.2, random_state=42)
print(f"Training data: {df_train['sentiment'].value_counts()}")
print(f"Testing data: {df_test['sentiment'].value_counts()}")

Training data: sentiment
negative    20039
positive    19961
Name: count, dtype: int64
Testing data: sentiment
positive    5039
negative    4961
Name: count, dtype: int64


## Clean the dataset
In the sentence making review comment, we need to convert all the sentences to a list of words. This list will be the sequence that we pass to the LSTM model. We will clean the dataset by:
* Remove `<br />` and other special characters.
* Remove punctuations
* Make everything lowercase
* Split across the space to get the list of words
* Remove stopwords, words that do not add any value to the sentence like 'the', 'a', 'an', etc.

In [132]:
# Functions to clean the text data
def remove_br(a_string):
    return a_string.replace("<br />", " ")

def remove_punctuation(a_string):
    clean_character_array = [character if character not in punctuation_set else "" for character in a_string]
    return "".join(clean_character_array)

def lower_case(a_string):
    return a_string.lower()

def split_words(a_string):
    # Split the string into words and get rid of extra white space
    return a_string.split()

In [133]:
test_string_original = "This is a test string. It has some punctuations! And some <br /> special characters."

test_string_cleaned = remove_br(test_string_original)
test_string_original, test_string_cleaned

('This is a test string. It has some punctuations! And some <br /> special characters.',
 'This is a test string. It has some punctuations! And some   special characters.')

In [134]:
punctuation_set = set(string.punctuation)

test_string_cleaned = remove_punctuation(test_string_cleaned)
test_string_original, test_string_cleaned


('This is a test string. It has some punctuations! And some <br /> special characters.',
 'This is a test string It has some punctuations And some   special characters')

In [135]:
test_string_cleaned = lower_case(test_string_cleaned)
test_string_original, test_string_cleaned

('This is a test string. It has some punctuations! And some <br /> special characters.',
 'this is a test string it has some punctuations and some   special characters')

In [136]:
test_list = split_words(test_string_cleaned)
test_string_original, test_list

('This is a test string. It has some punctuations! And some <br /> special characters.',
 ['this',
  'is',
  'a',
  'test',
  'string',
  'it',
  'has',
  'some',
  'punctuations',
  'and',
  'some',
  'special',
  'characters'])

In [137]:
# Remove stop words
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/zw5893/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [138]:
stop_words = set(stopwords.words('english'))

def no_more_stop_words(a_list):
    return [word for word in a_list if word not in stop_words]

test_list_cleaned = no_more_stop_words(test_list)
test_list, test_list_cleaned

(['this',
  'is',
  'a',
  'test',
  'string',
  'it',
  'has',
  'some',
  'punctuations',
  'and',
  'some',
  'special',
  'characters'],
 ['test', 'string', 'punctuations', 'special', 'characters'])

In [139]:
# This is a function which combines everything we have done so far in one function
def clean_string(a_string):
    return no_more_stop_words(split_words(lower_case(remove_punctuation(remove_br(a_string)))))

clean_string(test_string_original)

['test', 'string', 'punctuations', 'special', 'characters']

### Split the dataset

In [140]:
df_train["clean_review"] = df_train["review"].apply(clean_string)
df_test["clean_review"] = df_test["review"].apply(clean_string)
df_train.head()

,review,sentiment,clean_review
39087,That's what I kept asking myself during the ma...,negative,"[thats, kept, asking, many, fights, screaming,..."
30893,I did not watch the entire movie. I could not ...,negative,"[watch, entire, movie, could, watch, entire, m..."
45278,A touching love story reminiscent of In the M...,positive,"[touching, love, story, reminiscent, in, mood..."
16398,This latter-day Fulci schlocker is a totally a...,negative,"[latterday, fulci, schlocker, totally, abysmal..."
13653,"First of all, I firmly believe that Norwegian ...",negative,"[first, firmly, believe, norwegian, movies, co..."


### Convert labels from  negative and positive to 0 and 1

In [141]:
print(df_train["sentiment"].value_counts())
# adding 0 will convert boolean to integer
df_train["label"] = (df_train["sentiment"] == "positive") + 0
df_test["label"] = (df_test["sentiment"] == "positive") + 0

print(df_train["label"].value_counts())

sentiment
negative    20039
positive    19961
Name: count, dtype: int64
label
0    20039
1    19961
Name: count, dtype: int64


### Create embeddings
We will need to convert the reviews to numbers which the LSTM can understand. For this we will use word2vec.

In [142]:
# 1. Build Vocabulary
# Flatten all reviews into a single list of words
all_words = [word for review in df_train['clean_review'] for word in review]
word_counts = Counter(all_words)

# Map each word to a unique index
word_to_idx = {word: idx + 1 for idx, word in enumerate(word_counts.keys())}  # Start from 1 to reserve 0 for padding
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Vocabulary size
vocab_size = len(word_to_idx) + 1  # Add 1 for padding index


In [143]:
# 2. Convert Reviews to Indexed Sequences
indexed_reviews = [
    [word_to_idx[word] for word in review if word in word_to_idx] for review in df_train['clean_review']
]

In [144]:
padded_reviews = pad_sequence(
    [torch.tensor(review, dtype=torch.long) for review in indexed_reviews], 
    batch_first=True, 
    padding_value=0
)

# Convert labels to tensor
labels = torch.tensor(df_train["label"].values, dtype=torch.float32)

In [145]:
# Create a DataLoader for batching
batch_size = 64
dataset = TensorDataset(padded_reviews, labels)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [146]:
# 2. Define the LSTM Model
class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SentimentModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        out = self.fc(hidden.squeeze(0))
        return self.sigmoid(out)


In [147]:
# Hyperparameters
embedding_dim = 50
hidden_dim = 128
output_dim = 1  # Binary classification

# Initialize the model
model = SentimentModel(vocab_size, embedding_dim, hidden_dim, output_dim)
model = model.to(device)

In [148]:
# 3. Define Loss and Optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4. Train the Model with Batches
num_epochs = 5
model.train()


SentimentModel(
  (embedding): Embedding(147587, 50)
  (lstm): LSTM(50, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [150]:
for epoch in range(num_epochs):
    epoch_loss = 0
    # Wrap the DataLoader in tqdm for progress tracking
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}", leave=True)
    
    for batch_reviews, batch_labels in progress_bar:
        optimizer.zero_grad()  # Clear gradients
        batch_reviews, batch_labels = batch_reviews.to(device), batch_labels.to(device)

        # Forward pass
        predictions = model(batch_reviews).squeeze(1)
        
        # Compute loss
        loss = criterion(predictions, batch_labels)
        
        # Backward pass and optimize
        loss.backward()
        optimizer.step()
        
        # Accumulate loss
        epoch_loss += loss.item()
        
        # Update progress bar with current loss
        progress_bar.set_postfix({"Batch Loss": loss.item()})
    
    print(f"Epoch {epoch + 1} completed with Loss: {epoch_loss:.4f}")

Epoch 1/5: 100%|██████████| 625/625 [00:40<00:00, 15.46it/s, Batch Loss=0.694]


Epoch 1 completed with Loss: 433.5767


Epoch 2/5: 100%|██████████| 625/625 [00:38<00:00, 16.22it/s, Batch Loss=0.692]


Epoch 2 completed with Loss: 433.2970


Epoch 3/5: 100%|██████████| 625/625 [00:38<00:00, 16.27it/s, Batch Loss=0.693]


Epoch 3 completed with Loss: 433.2699


Epoch 4/5: 100%|██████████| 625/625 [00:38<00:00, 16.23it/s, Batch Loss=0.693]


Epoch 4 completed with Loss: 433.2658


Epoch 5/5: 100%|██████████| 625/625 [00:38<00:00, 16.21it/s, Batch Loss=0.694]

Epoch 5 completed with Loss: 433.2471


I probably screwed something up. I will try to fix it in the next version.